<a href="https://colab.research.google.com/github/Eunchae-L/ADD2023-SSAD-PASE/blob/main/ADD2023_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%cd /content/drive/MyDrive/STUDY/Project/ADD2023

/content/drive/MyDrive/STUDY/Project/ADD2023


In [2]:
import os
import glob
import numpy as np
import random
from tqdm import tqdm
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader

import soundfile as sf
import librosa

import pickle

# Config

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_CHUNKED_DIR = "data/chunked/stage2/train"
DEV_CHUNKED_DIR = "data/chunked/stage2/dev"

CACHE_TRAIN_DIR = 'finetuning/cached_z/train'
CACHE_DEV_DIR = 'finetuning/cached_z/dev'
CACHE_TEST_DIR = 'finetuning/cached_z/test'

ENCODER_CKPT = "checkpoints/pretrain_epoch100.pt"

BATCH_SIZE = 32
NUM_WORKERS = 0

LABEL_MAP = {
    "genuine": 0,
    "fake": 1
}

# Z Caching

In [4]:
class ChunkedDataset(Dataset):
  def __init__(self, root_dir):
    self.root_dir = root_dir
    self.files = []

    npy_files = glob.glob(os.path.join(self.root_dir, "*.npy"))
    for f in npy_files:
        base = os.path.basename(f)

        # label parsing from filename
        # expected: "... .wav_genuine_..." or "... .wav_fake_..."
        if ".wav_" not in base:
            continue
        tail = base.split(".wav_", 1)[1].lower()

        if tail.startswith("train_fake_"):
            label = 1
        elif tail.startswith("train_genuine_"):
            label = 0
        elif tail.startswith("dev_fake_"):
            label = 1
        elif tail.startswith("dev_genuine_"):
            label = 0
        else:
            continue

        self.files.append((f, label))

  def __len__(self):
    return len(self.files)

  def __getitem__(self, idx):
    path, label = self.files[idx]
    audio = np.load(path)
    audio = torch.tensor(audio).float()
    return audio, label, path

# Encoder load

In [5]:
CHANNEL_LIST = (16,32,64,128,128,256,256,512)
KERNEL_LIST = (10,8,8,4,4,4,4,4)
STRIDE_LIST = (5,4,2,2,2,2,2,2)

embedding_dim = 512
worker_input_dim = 256
proj_hidden = 512

TCN_LAYERS = 6
# DROPOUT = 0.0

In [6]:
class ConvBlock(nn.Module):
  def __init__(
      self,
      in_ch,
      out_ch,
      kernel_size,
      stride,
      # dropout=0.0
  ):
      super().__init__()
      padding = kernel_size // 2

      self.conv = nn.Conv1d(
          in_ch,
          out_ch,
          kernel_size=kernel_size,
          stride=stride,
          padding=padding
      )

      self.bn = nn.BatchNorm1d(out_ch)
      self.act = nn.ReLU()
      #dropout 값이 0보다 크면 Dropout 레이어를 쓰고,0이면 아무 일도 하지 않는 Identity 레이어를 쓴다.
      # self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

  def forward(self, x):
      x = self.conv(x)
      x = self.bn(x)
      x = self.act(x)
      # x = self.dropout(x)
      return x

class TCNBlock(nn.Module):
  def __init__(
      self,
      channels,
      dilation,
      kernel_size,
      causal=False
  ):
      super().__init__()
      self.causal = causal

      if causal:
          padding = (kernel_size - 1) * dilation
      else:
          padding = ((kernel_size - 1) * dilation) // 2

      self.conv = nn.Conv1d(
          channels,
          channels,
          kernel_size=kernel_size,
          dilation=dilation,
          padding=padding
      )
      self.bn = nn.BatchNorm1d(channels)
      self.act = nn.ReLU()

  def forward(self, x):
      out = self.conv(x)

      if self.causal:
          out = out[..., : x.size(-1)] #causal crop
      out = self.bn(out)
      out = self.act(out)
      return x + out

In [7]:
class Encoder(nn.Module):
  def __init__(
      self,
      channel_list,
      kernel_list,
      stride_list,
      embedding_dim=512,
      tcn_layers=4,
      tcn_kernel_size=3,
      tcn_causal=False,
      proj_hidden=512,
      worker_input_dim=256,
  ):
      super().__init__()

      assert len(channel_list) == 8
      assert len(kernel_list) == 8
      assert len(stride_list) == 8

      self.embedding_dim = embedding_dim
      self.worker_input_dim = worker_input_dim

      # 8 Conv1D Block
      convs = []
      in_ch = 1
      for out_ch, kernel_size, stride in zip(channel_list, kernel_list, stride_list):
          convs.append(
              ConvBlock(
                  in_ch,
                  out_ch,
                  kernel_size=kernel_size,
                  stride=stride,
                  # dropout=dropout
              )
          )
          in_ch = out_ch
      self.conv_blocks = nn.ModuleList(convs) #nn.ModuleList: 모듈들을 리스트 형태로 관리, 동적으로 모듈 추가/삭제 가능

      # skip projection
      self.skip_proj = nn.ModuleList([
          nn.Conv1d(ch, embedding_dim, kernel_size=1)
          for ch in channel_list
      ])

      # TCN
      self.tcn = nn.Sequential(*[
          TCNBlock(
              channels=embedding_dim,
              dilation=2 ** i,
              kernel_size=tcn_kernel_size,
              causal=tcn_causal
          )
          for i in range(tcn_layers)
      ])

      # Nonlinear projection head
      self.proj_head = nn.Sequential(
          nn.Conv1d(embedding_dim, proj_hidden, kernel_size=1),
          nn.Tanh(),
          nn.Conv1d(proj_hidden, worker_input_dim, kernel_size=1)
      )

      self.bn = nn.BatchNorm1d(worker_input_dim)

  def forward(self, x):
      """
      x: (B, T) or (B, 1, T)
      return: (B, worker_input_dim, T)
      """
      if x.dim() == 2:
        x = x.unsqueeze(1)

      skip_feats = []
      out = x

      for block, proj in zip(self.conv_blocks, self.skip_proj):
        out = block(out)
        skip_feats.append(proj(out))

      #temporal alignment (crop to shortest)
      min_t = min(f.size(-1) for f in skip_feats)
      skip_feats = [f[..., :min_t] for f in skip_feats]

      z = torch.stack(skip_feats, dim=0).sum(dim=0)

      z = self.tcn(z)
      z = self.proj_head(z)
      z = self.bn(z)

      return z

In [8]:
def load_pretrained_encoder(ckpt_path, device):
  ckpt = torch.load(ckpt_path, map_location=device)

  cfg = ckpt["config"]
  encoder = Encoder(
    channel_list=cfg["channel_list"],
    kernel_list=cfg["kernel_list"],
    stride_list=cfg["stride_list"],
    embedding_dim=embedding_dim,
    worker_input_dim=worker_input_dim,
    proj_hidden=proj_hidden,
    tcn_layers=cfg["tcn_layers"],
  ).to(device)

  encoder.load_state_dict(ckpt["encoder"], strict=False)
  encoder.eval()

  return encoder

In [9]:
encoder = load_pretrained_encoder(ENCODER_CKPT, DEVICE)

KeyError: 'config'

# Feature Extraction

In [ ]:
@torch.no_grad()
def feature_extraction(
    dataset,
    save_dir,
    encoder,
    device,
    batch_size=32,
    num_workers=2,
    split_name="train",
    pin_memory=False
):
  os.makedirs(save_dir, exist_ok=True)
  encoder.eval()
  loader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=False,
      num_workers=num_workers,
      pin_memory=pin_memory,
      persistent_workers=(num_workers > 0)
  )
  meta = []

  for audio, label, path in tqdm(loader, desc=f"Caching {split_name}"):
      audio = audio.to(device, non_blocking=pin_memory)
      # Encoder forward
      # Expected output: (B, C, T)
      z = encoder(audio)
      z = z.mean(dim=-1) #(B, C)

      z = z.cpu().numpy()
      label = torch.as_tensor(label).cpu().numpy()

      for i in range(z.shape[0]):
        base = os.path.basename(path[i])
        name = os.path.splitext(base)[0]
        z_path = os.path.join(save_dir, f"{name}.npy")
        np.save(z_path, z[i])
        meta.append((z_path, int(label[i])))

  #save metadata
  meta_path = os.path.join(save_dir, f"meta_{split_name}.csv")
  with open(meta_path, "w") as f:
    f.write("path,label\n")
    for p, l in meta:
      f.write(f"{p},{l}\n")

  print(f"Cached {len(meta)}features saved to {save_dir}")

In [ ]:
# train_meta = pd.read_csv("data/Train/train_metadata.csv")
# dev_meta = pd.read_csv("data/Dev/dev_metadata.csv")
# stage2_meta = pd.concat([train_meta, dev_meta], axis=0)

# train_set = ChunkedDataset(TRAIN_CHUNKED_DIR)
# dev_set = ChunkedDataset(DEV_CHUNKED_DIR)
# print(f"Train samples: {len(train_set)}")
# print(f"Dev samples : {len(dev_set)}")


# feature_extraction(
#     train_set,
#     CACHE_TRAIN_DIR,
#     encoder=encoder,
#     batch_size=BATCH_SIZE,
#     num_workers=NUM_WORKERS,
#     device=DEVICE,
#     split_name="train"
# )
# feature_extraction(
#     dev_set,
#     CACHE_DEV_DIR,
#     encoder=encoder,
#     batch_size=BATCH_SIZE,
#     num_workers=NUM_WORKERS,
#     device=DEVICE,
#     split_name="dev"
# )
# print("[DONE] Encoder representation caching complete.")

# Classifier

저장된 z 불러오기

In [ ]:
def z_to_image(z: torch.Tensor, h=16, w=16):
  """
  z: (C,) where C == h*w (256)
  return: (1, H, W)
  """
  if z.numel() != h*w:
    raise ValueError(f"Invalid z shape: {z.shape} at {z.path}")
  return z.view(1, h, w)

-> 캐싱된 z는 (C,) 벡터(예: 256-d)이고, LCNN/SENet은 원래 2D 입력(스펙트럼) 기반이라서 z를 2D로 reshape 해서 CNN에 넣는 어댑터를 같이 제공

256 -> 1 * 16 * 16

In [ ]:
class CachedZDataset(Dataset):
  """
  Dataset for Stage2 classifier training using cached encoder representations

  Each sample:
  - z: (C, T')
  - label: 0(genuine) , 1(fake)

  Meta CSV format:
  path, label
  data/.../xxx.npy, 0
  """

  def __init__(self, meta_csv, as_image=True, image_hw=(16,16)):
    self.as_image = as_image
    self.h, self.w = image_hw

    items = []

    df = meta_csv
    for p, y in zip(df["path"].tolist(), df["label"].tolist()):
      items.append((p, int(y)))

    if not items:
      raise ValueError("No cached z files found. Check meta_csv/root_dir layout.")

    items.sort()
    self.items = items

  def __len__(self):
    return len(self.items)

  def __getitem__(self, idx):
    path, y = self.items[idx]
    z = np.load(path).astype(np.float32)
    z = torch.from_numpy(z)

    if self.as_image:
      x = z_to_image(z, h=self.h, w=self.w)
    else:
      x = z #(C,)

    return x, int(y), path

In [ ]:
def make_loader(dataset, batch_size=32, num_workers=2, shuffle=True, pin_memory=True):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=True
)

In [ ]:
cached_train_meta = pd.read_csv("finetuning/cached_z/train/meta_train.csv")
cached_dev_meta = pd.read_csv("finetuning/cached_z/dev/meta_dev.csv")
cached_stage2_meta = pd.concat([cached_train_meta, cached_dev_meta], axis=0)
train_ds = CachedZDataset(meta_csv=cached_stage2_meta, as_image=True, image_hw=(16,16))
dev_ds = CachedZDataset(meta_csv=cached_stage2_meta, as_image=True, image_hw=(16,16))

train_loader = make_loader(train_ds, batch_size=32, num_workers=0, shuffle=True)
dev_loader = make_loader(dev_ds, batch_size=32, num_workers=0, shuffle=False)

# Classifier

:LCNN-big/LCNN-small/SENet12

MFM, 초기화, SE 블록

In [ ]:
class MFM(nn.Module):
    def forward(self, x):
        c = x.size(1)
        assert c % 2 == 0, f"MFM expects even channels, got {c}"
        a, b = torch.split(x, c // 2, dim=1)
        return torch.max(a, b)

def kaiming_init(m: nn.Module):
    if isinstance(m, (nn.Conv2d, nn.Linear)) and not isinstance(m, nn.LazyLinear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, hidden, kernel_size=1, bias=True)
        self.fc2 = nn.Conv2d(hidden, channels, kernel_size=1, bias=True)

    def forward(self, x):
        s = self.pool(x)
        s = F.relu(self.fc1(s), inplace=True)
        s = torch.sigmoid(self.fc2(s))
        return x * s

LCNN-big

In [ ]:
class LCNNBig(nn.Module):
    """
    - 입력: (B, 1, H, W)
    """
    def __init__(self, num_classes=2, dropout=0.75):
        super().__init__()
        self.mfm = MFM()

        self.conv1 = nn.Conv2d(1, 64, kernel_size=5, stride=1, padding=2)       # -> MFM -> 32
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv4 = nn.Conv2d(32, 64, kernel_size=1, stride=1, padding=0)      # -> MFM -> 32
        self.bn6   = nn.BatchNorm2d(32)
        self.conv7 = nn.Conv2d(32, 96, kernel_size=3, stride=1, padding=1)      # -> MFM -> 48
        self.pool9 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bn10  = nn.BatchNorm2d(48)

        self.conv11 = nn.Conv2d(48, 96, kernel_size=1, stride=1, padding=0)     # -> MFM -> 48
        self.bn13   = nn.BatchNorm2d(48)
        self.conv14 = nn.Conv2d(48, 128, kernel_size=3, stride=1, padding=1)    # -> MFM -> 64
        self.pool16 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv17 = nn.Conv2d(64, 128, kernel_size=1, stride=1, padding=0)    # -> MFM -> 64
        self.bn19   = nn.BatchNorm2d(64)
        self.conv20 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)     # -> MFM -> 32
        self.bn22   = nn.BatchNorm2d(32)
        self.conv23 = nn.Conv2d(32, 64, kernel_size=1, stride=1, padding=0)     # -> MFM -> 32
        self.bn25   = nn.BatchNorm2d(32)
        self.conv26 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)     # -> MFM -> 32
        self.pool28 = nn.MaxPool2d(kernel_size=2, stride=2)

        # FC part: use LazyLinear so input spatial size can vary
        self.fc29   = nn.LazyLinear(160)
        self.bn31   = nn.BatchNorm1d(80)
        self.drop   = nn.Dropout(p=dropout)
        self.fc32   = nn.Linear(80, num_classes)

        self.apply(kaiming_init)

    def extract_feat(self, x):
        # x: (B,1,H,W)
        x = self.mfm(self.conv1(x))       # 64 -> 32
        x = self.pool3(x)

        x = self.mfm(self.conv4(x))       # 64 -> 32
        x = self.bn6(x)
        x = self.mfm(self.conv7(x))       # 96 -> 48
        x = self.pool9(x)
        x = self.bn10(x)

        x = self.mfm(self.conv11(x))      # 96 -> 48
        x = self.bn13(x)
        x = self.mfm(self.conv14(x))      # 128 -> 64
        x = self.pool16(x)

        x = self.mfm(self.conv17(x))      # 128 -> 64
        x = self.bn19(x)
        x = self.mfm(self.conv20(x))      # 64 -> 32
        x = self.bn22(x)
        x = self.mfm(self.conv23(x))      # 64 -> 32
        x = self.bn25(x)
        x = self.mfm(self.conv26(x))      # 64 -> 32
        x = self.pool28(x)

        x = torch.flatten(x, 1)           # (B, *)
        x = self.fc29(x)                  # (B,160)
        # MFM on vector: split channels 160->80
        a, b = torch.split(x, 80, dim=1)
        x = torch.max(a, b)           # (B,80)

        x = self.bn31(x)
        x = self.drop(x)
        return x # embedding (B,80)

    def forward(self, x):
        return self.fc32(self.extract_feat(x))

입력 크기 독립적으로 동작하도록 LazyLinear 사용

LCNN-small

In [ ]:
class LCNNSmall(nn.Module):
    def __init__(self, num_classes=2, dropout=0.75):
        super().__init__()
        self.mfm = MFM()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, stride=1, padding=2)       # -> MFM -> 8
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv4 = nn.Conv2d(8, 16, kernel_size=1, stride=1, padding=0)       # -> MFM -> 8
        self.bn6   = nn.BatchNorm2d(8)
        self.conv7 = nn.Conv2d(8, 24, kernel_size=3, stride=1, padding=1)       # -> MFM -> 12
        self.pool9 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bn10  = nn.BatchNorm2d(12)

        self.conv11 = nn.Conv2d(12, 24, kernel_size=1, stride=1, padding=0)     # -> MFM -> 12
        self.bn13   = nn.BatchNorm2d(12)
        self.conv14 = nn.Conv2d(12, 24, kernel_size=3, stride=1, padding=1)     # -> MFM -> 12
        self.pool16 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv17 = nn.Conv2d(12, 24, kernel_size=1, stride=1, padding=0)     # -> MFM -> 12
        self.bn19   = nn.BatchNorm2d(12)
        self.conv20 = nn.Conv2d(12, 8, kernel_size=3, stride=1, padding=1)      # -> MFM -> 4
        self.bn22   = nn.BatchNorm2d(4)
        self.conv23 = nn.Conv2d(4, 8, kernel_size=1, stride=1, padding=0)       # -> MFM -> 4
        self.bn25   = nn.BatchNorm2d(4)
        self.conv26 = nn.Conv2d(4, 8, kernel_size=3, stride=1, padding=1)       # -> MFM -> 4
        self.pool28 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc29   = nn.LazyLinear(64)
        self.bn31   = nn.BatchNorm1d(32)
        self.drop   = nn.Dropout(p=dropout)
        self.fc32   = nn.Linear(32, num_classes)

        self.apply(kaiming_init)

    def extract_feat(self, x):
        x = self.mfm(self.conv1(x))       # 16 -> 8
        x = self.pool3(x)

        x = self.mfm(self.conv4(x))       # 16 -> 8
        x = self.bn6(x)
        x = self.mfm(self.conv7(x))       # 24 -> 12
        x = self.pool9(x)
        x = self.bn10(x)

        x = self.mfm(self.conv11(x))      # 24 -> 12
        x = self.bn13(x)
        x = self.mfm(self.conv14(x))      # 24 -> 12
        x = self.pool16(x)

        x = self.mfm(self.conv17(x))      # 24 -> 12
        x = self.bn19(x)
        x = self.mfm(self.conv20(x))      # 8 -> 4
        x = self.bn22(x)
        x = self.mfm(self.conv23(x))      # 8 -> 4
        x = self.bn25(x)
        x = self.mfm(self.conv26(x))      # 8 -> 4
        x = self.pool28(x)

        x = torch.flatten(x, 1)
        x = self.fc29(x)                  # (B,64)

        a, b = torch.split(x, 32, dim=1)
        x = torch.max(a, b)           # (B,32)

        x = self.bn31(x)
        x = self.drop(x)
        return x  # embedding (B,32)

    def forward(self, x):
        return self.fc32(self.extract_feat(x))

채널 수만 축소, FC는 64->MFM->32->2

SENet12

In [ ]:
class SEBasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.se    = SEBlock(out_ch, reduction=reduction)

        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = self.se(out)

        if self.downsample is not None:
            identity = self.downsample(identity)

        out = F.relu(out + identity, inplace=True)
        return out

class SENet12(nn.Module):
    def __init__(self, num_classes=2, reduction=16):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.layer1 = self._make_layer(16, 16, blocks=1, stride=1, reduction=reduction)
        self.layer2 = self._make_layer(16, 32, blocks=2, stride=2, reduction=reduction)
        self.layer3 = self._make_layer(32, 64, blocks=3, stride=2, reduction=reduction)
        self.layer4 = self._make_layer(64, 128, blocks=1, stride=2, reduction=reduction)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(128, num_classes)

        self.apply(kaiming_init)

    def _make_layer(self, in_ch, out_ch, blocks, stride, reduction):
        layers = [SEBasicBlock(in_ch, out_ch, stride=stride, reduction=reduction)]
        for _ in range(1, blocks):
            layers.append(SEBasicBlock(out_ch, out_ch, stride=1, reduction=reduction))
        return nn.Sequential(*layers)

    def extract_feat(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return x

    def forward(self, x):
        x = self.extract_feat(x)
        x = self.fc(x)
        return x

A-Softmax (SphereFace) 레이어 + 모델 래퍼

In [ ]:
import math
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# -------------------------
# A-Softmax (SphereFace) head
# -------------------------
class ASoftmaxLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, m: int = 4):
        super().__init__()
        assert m in [1, 2, 3, 4, 5], "Implementing m up to 5 is typical; m=4 recommended here."
        self.in_features = in_features
        self.out_features = out_features
        self.m = m
        self.W = nn.Parameter(torch.randn(out_features, in_features))
        nn.init.xavier_normal_(self.W)

    def _cos_m_theta(self, cos_theta: torch.Tensor):
        # cos(mθ) using multiple-angle formulas (Chebyshev polynomials)
        m = self.m
        if m == 1:
            return cos_theta
        elif m == 2:
            return 2*cos_theta**2 - 1
        elif m == 3:
            return 4*cos_theta**3 - 3*cos_theta
        elif m == 4:
            return 8*cos_theta**4 - 8*cos_theta**2 + 1
        elif m == 5:
            return 16*cos_theta**5 - 20*cos_theta**3 + 5*cos_theta
        else:
            raise ValueError("Unsupported m")

    def forward(self, x: torch.Tensor, y: torch.Tensor):
        """
        x: (B, in_features)
        y: (B,)
        returns: logits (B, out_features)
        """
        # normalize
        x_norm = torch.norm(x, p=2, dim=1, keepdim=True).clamp_min(1e-8)  # (B,1)
        x_hat = x / x_norm                                                # (B,F)

        W_hat = F.normalize(self.W, p=2, dim=1)                            # (C,F)

        cos_theta = torch.matmul(x_hat, W_hat.t()).clamp(-1.0, 1.0)        # (B,C)
        cos_m_theta = self._cos_m_theta(cos_theta)                         # (B,C)

        theta = torch.acos(cos_theta)                                      # (B,C)
        k = torch.floor(self.m * theta / math.pi)                          # (B,C)
        phi_theta = ((-1.0)**k) * cos_m_theta - 2.0*k

        # scale by ||x||
        logits = cos_theta * x_norm                                         # (B,C)
        # replace target logits with phi * ||x||
        idx = torch.arange(x.size(0), device=x.device)
        logits[idx, y] = (phi_theta[idx, y] * x_norm[idx, 0])

        return logits


@torch.no_grad()
def bonafide_cosine_score(features: torch.Tensor, W: torch.Tensor, bonafide_class: int = 0):
    """
    features: (B, F)
    W: (num_classes, F) unnormalized weights from last layer/head
    score = cosine(feature, W_bonafide)
    """
    f = F.normalize(features, p=2, dim=1)
    w = F.normalize(W[bonafide_class:bonafide_class+1], p=2, dim=1)  # (1,F)
    return (f * w).sum(dim=1)  # (B,)


class LCNNForASoftmax(nn.Module):
    """
    base_lcnn should output embedding vector of shape (B, embed_dim)
    """
    def __init__(self, base_lcnn: nn.Module, embed_dim: int, num_classes: int = 2, m: int = 4):
        super().__init__()
        self.backbone = base_lcnn
        self.head = ASoftmaxLinear(embed_dim, num_classes, m=m)

    def forward(self, x, y=None, return_feat=False):
        feat = self.backbone.extract_feat(x)  # (B, embed_dim)
        if return_feat:
            return feat
        assert y is not None, "y is required for A-Softmax forward"
        logits = self.head(feat, y)
        return logits

EER 계산 + 평가 함수

In [ ]:
def compute_eer(labels, scores):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores).astype(float)

    # sort by score descending
    idx = np.argsort(scores)[::-1]
    labels = labels[idx]
    scores = scores[idx]

    # positives = genuine (label=0), negatives = fake (label=1)
    pos = (labels == 0).astype(int)
    neg = (labels == 1).astype(int)

    P = pos.sum()
    N = neg.sum()
    if P == 0 or N == 0:
        return 1.0, 0.0

    # Sweep threshold at each score
    # Accept genuine if score >= thr
    tp = np.cumsum(pos)
    fp = np.cumsum(neg)

    fn = P - tp
    tn = N - fp

    far = fp / (fp + tn + 1e-12)   # false accept rate: fake accepted as genuine
    frr = fn / (fn + tp + 1e-12)   # false reject rate: genuine rejected

    i = np.argmin(np.abs(far - frr))
    eer = 0.5 * (far[i] + frr[i])
    thr = scores[i]
    return float(eer), float(thr)


@torch.no_grad()
def evaluate_eer(model, loader, device, mode: str):
    """
    mode: 'lcnn' or 'senet'
    """
    model.eval()
    all_labels, all_scores = [], []

    for x, y, _ in loader:
        x = x.to(device)
        y = torch.as_tensor(y, device=device).long()

        if mode == "senet":
            feat = model.extract_feat(x)           # (B,128)
            W = model.fc.weight                    # (2,128)
            score = bonafide_cosine_score(feat, W, bonafide_class=0)
        else:
            # LCNN wrapper
            feat = model(x, return_feat=True)      # (B,embed)
            W = model.head.W                       # (2,embed)
            score = bonafide_cosine_score(feat, W, bonafide_class=0)

        all_labels.extend(y.detach().cpu().tolist())
        all_scores.extend(score.detach().cpu().tolist())

    eer, thr = compute_eer(all_labels, all_scores)
    return eer, thr

Option A 훈련 루프 (early stopping + best ckpt)

In [ ]:
def train_option_a(
    model,
    train_loader,
    dev_loader,
    device,
    mode: str,
    epochs: int = 80,
    lr: float = 1e-3,
    weight_decay: float = 1e-3,
    patience: int = 10,
    ckpt_dir: str = "checkpoints/stage2_optionA_final",
    ckpt_name: str = "best.pt"
):
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = os.path.join(ckpt_dir, ckpt_name)

    model.to(device)
    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        betas=(0.9, 0.999),
        weight_decay=weight_decay
    )
    criterion = nn.CrossEntropyLoss()

    best_eer = float("inf")
    best_epoch = -1
    bad = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        n = 0

        pbar = tqdm(train_loader, desc=f"[Train] Epoch {epoch}/{epochs}", leave=True)
        for x, y, _ in pbar:
            x = x.to(device)
            y = torch.as_tensor(y, device=device).long()

            optimizer.zero_grad(set_to_none=True)

            if mode == "senet":
                logits = model(x)                     # (B,2)
            else:
                logits = model(x, y=y)                # (B,2) from A-Softmax head

            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            n += bs
            pbar.set_postfix(loss=total_loss / max(n, 1))

        # ---- dev EER ----
        dev_eer, dev_thr = evaluate_eer(model, dev_loader, device, mode=mode)
        print(f"[Dev] epoch={epoch}  EER={dev_eer:.5f}  thr={dev_thr:.5f}")

        # ---- early stopping ----
        improved = dev_eer < best_eer
        if improved:
            best_eer = dev_eer
            best_epoch = epoch
            bad = 0
            torch.save(
                {
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "epoch": epoch,
                    "best_eer": best_eer,
                    "mode": mode,
                },
                ckpt_path
            )
            print(f"[CKPT] saved best -> {ckpt_path} (EER={best_eer:.5f})")
        else:
            bad += 1
            if bad >= patience:
                print(f"[EarlyStop] no improvement for {patience} epochs. best_epoch={best_epoch}, best_eer={best_eer:.5f}")
                break

    print(f"[DONE] best_epoch={best_epoch}, best_eer={best_eer:.5f}")
    return ckpt_path


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- LCNN-big (A-Softmax) ----
base = LCNNBig(dropout=0.75)
model = LCNNForASoftmax(base_lcnn=base, embed_dim=80, num_classes=2, m=4)

train_option_a(
    model=model,
    train_loader=train_loader,
    dev_loader=dev_loader,
    device=DEVICE,
    mode="lcnn",
    epochs=80,
    lr=1e-3,
    weight_decay=1e-3,
    patience=10,
    ckpt_dir="checkpoints/stage2_optionA_final",
    ckpt_name="lcnn_big_best.pt"
)

# ---- LCNN-small (A-Softmax) ----
base = LCNNSmall(dropout=0.75)
model = LCNNForASoftmax(base_lcnn=base, embed_dim=32, num_classes=2, m=4)

train_option_a(
    model=model,
    train_loader=train_loader,
    dev_loader=dev_loader,
    device=DEVICE,
    mode="lcnn",
    epochs=80,
    lr=1e-3,
    weight_decay=1e-3,
    patience=10,
    ckpt_dir="checkpoints/stage2_optionA_final",
    ckpt_name="lcnn_small_best.pt"
)

# ---- SENet12 (CrossEntropy) ----
model = SENet12(num_classes=2, reduction=16)

train_option_a(
    model=model,
    train_loader=train_loader,
    dev_loader=dev_loader,
    device=DEVICE,
    mode="senet",
    epochs=80,
    lr=1e-3,
    weight_decay=1e-3,
    patience=10,
    ckpt_dir="checkpoints/stage2_optionA_final",
    ckpt_name="senet12_best.pt"
)